# Potencial Gravitacional de un Disco Protoplanetario

Santiago Andrés Acosta Díaz

La idea de este ejercicio es resolver la PDE 

$$
    \frac{\partial^2 \Phi}{\partial r^2} + \frac{1}{r} \frac{\partial \Phi}{\partial r} + \frac{\partial^2 \Phi}{\partial z^2} = 4 \pi G \rho(r, z) 
$$

con

$$
    \rho(r, z) = \rho_0 e^{-\frac{r}{r_0}} e^{-\frac{z^2}{2h^2}}
$$

donde 

$$
    \rho_0 = 10^{-9} \,g \, cm^{-3}, \qquad h = 0.5 \,UA \qquad r_0 = 3 \, UA
$$

y condiciones de Dirichlet $\Phi = 0$ en los bordes $R_{max} = 10 \, UA$ y $Z_{max} = 2 \, UA$.

In [13]:
import numpy as np
import matplotlib.pyplot as plt


# -----------------------------
#    CONSTANTES
# -----------------------------

rho0 = 10**(-9)
hc = 0.5
r0 = 3.0

Rmax = 10.0
Zmax = 2.0

Nr = 50
Nz = 20

G = 6.674e-8    # G pero en g y cm para que tenga las mismas unidades que rho_0

RHSconstant = 4. * np.pi * G
rExpConstant = -1./(r0)   # Constante dentro de la exp de r
zExpConstant = -1./(2. * hc**2)   # Constante dentro de la exp de z

# -----------------------------
#    OTRAS DEFINICIONES
# -----------------------------

def RHS(r, z):
    """ Lado derecho de la ecuación de Poisson"""
    return RHSconstant * np.exp(-r/r0) * np.exp(rExpConstant * r) * np.exp(zExpConstant * z**2)

## Discretización

Lo primero es discretizar la PDE. El primer problema es saber si tenemos que hacer algo especial por estar usando cilíndricas, sin embargo, para el método que estamos usando y sobre todo, considerando que estamos trabajando las coordenadas naturalmente cartesianas (no involucramos el ángulo azimutal), entonces podemos discretizar por diferencias finitas como siempre lo hemos hecho.

Considerando que 

$$
    \frac{\partial^2 f}{\partial x^2} \approx \frac{f_{i-1} - 2f_{i} + f_{i+1}}{\Delta x^2}
$$

y que 

$$
    \frac{\partial f}{\partial x} \approx \frac{f_{i+1} - f_{i-1}}{2 \Delta x},
$$

reemplazando en la ecuación, y suponiendo que en efecto espaciamos el plano r-z igualmente ($\Delta r = \Delta z = h$), podemos llegar a que 

$$
    \frac{1}{h^2} \left( \Phi_{i-1, j} + \Phi_{i+ 1, j} + \Phi_{i, j-1} + \Phi_{i, j+1} - 4\Phi_{i, j} \right) + \frac{1}{2 r_i h} \left( \Phi_{i+1, j} - \Phi_{i-1, j} \right) = RHS
$$

con $RHS = 2 \pi G \rho(r, z)$.


__CONSIDERACIÓN:__ puede que el primer término en $r = 0$ sea problemática por el $1/r$ , sin embargo, nosotros evitamos tocar este bien porque las condiciones de frontera nos establecen que $\Phi$ en los bordes es $0$. 

## Gauss-Seidel

Básicamente, tenemos que despejar los términos $\Phi_{i, j}$ y a partir de ellos, vamos a tener nuestra nueva regla de iteración.

$$
    \Phi_{i, j}^{(new)} = \frac{1}{4} \left[ \Phi_{i-1, j} + \Phi_{i+1, j} + \Phi_{i, j-1} + \Phi_{i, j+1} + \frac{h}{2r_i} \left( \Phi_{i+1, j} - \Phi_{i-1, j} \right) - h^2 RHS \right]
$$

In [35]:
def Gauss_Seidel(r, z, PHI):
    """ Función que toma la mesh en un instante dado de tiempo y actualiza con Gauss-Seidel puro"""

    # obtenemos los shapes para hacer las iteraciones
    Rshape = r.shape[0]
    Zshape = z.shape[0]

    # Como las condiciones de frontera me dejan en 0.0 todos los bordes, iteramos en el interior
    for i in range(1, Rshape - 1):
        for j in range(1, Zshape - 1):

            PHI[i, j] = 0.25 * ( PHI[i-1, j] + PHI[i+1, j] + PHI[i, j-1] + PHI[i, j+1] 
                                 + (hc / (2. * r[i]) )*( PHI[i+1, j] - PHI[i-1, j] ) - hc**2 * RHS(r[i], z[j]) )

    return PHI

            
    
    

In [37]:
r = np.linspace(0, Rmax, Nr)
z = np.linspace(-Zmax, Zmax, Nz)

PHI = np.zeros([Nr, Nz])

PHI = Gauss_Seidel(r, z, PHI) # This 


[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -7.56796063e-11 -3.31478283e-10 -1.16402815e-09
  -3.42323679e-09 -8.45598764e-09 -1.75594243e-08 -3.06793386e-08
  -4.51472912e-08 -5.60338455e-08 -5.87554841e-08 -5.21663276e-08
  -3.93310644e-08 -2.52781935e-08 -1.39197268e-08 -6.61216146e-09
  -2.73419894e-09 -9.96108116e-10 -3.24706635e-10  0.00000000e+00]
 [ 0.00000000e+00 -7.33842538e-11 -3.23257455e-10 -1.13720953e-09
  -3.34972333e-09 -8.29000338e-09 -1.72542639e-08 -3.02309593e-08
  -4.46415253e-08 -5.56436437e-08 -5.86578321e-08 -5.24282126e-08
  -3.98625824e-08 -2.58951645e-08 -1.44556634e-08 -6.98826388e-09
  -2.95557143e-09 -1.10819027e-09 -3.74556314e-10  0.00000000e+00]
 [ 0.00